# Calculons les scores avec une couche

Faire les produits, ajouter le biais et appliquer la même règle à chaque extrait.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#construire-un-module).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La retouche de B nous a donné une décision correcte, mais aucune règle pour un nouvel extrait. Reprenons les scores initiaux et calculons-les depuis les mesures. Une couche linéaire multiplie celles-ci par des poids, additionne les produits, puis ajoute un biais.

| Score calculé | Poids RMS | Poids facteur de crête | Biais |
| --- | --- | --- | --- |
| Voix | 0 | 1 | 0 |
| Percussion | 0 | −1 | 0 |

Nous ignorons provisoirement la RMS pour refaire les calculs de tête. Le poids zéro est un choix pédagogique, pas une conclusion sur son utilité pour reconnaître des sons.

Pour A, le score voix vaut 0×0,10 + 1×1,5 + 0 = 1,5. Le score percussion vaut 0×0,10 − 1×1,5 + 0 = −1,5. C’est exactement la première ligne du chapitre précédent.

Dans PyTorch, nn désigne le paquet torch.nn, qui fournit les couches. nn.Linear(2,2) reçoit deux mesures par ligne et produit deux scores. Sa matrice weight possède une ligne par sortie ; bias possède une valeur par sortie.

Linear choisit normalement des paramètres initiaux aléatoires. Le programme complet les remplace par notre tableau : copy_ écrit les poids et zero_ annule les biais. no_grad évite de préparer le calcul des dérivées pendant cette installation. Nous expliquerons ces dérivées lorsque le modèle devra apprendre.

Un seul appel applique la même règle aux trois lignes. Les scores sont donc [[1,5 ; −1,5], [4,5 ; −4,5], [2,5 ; −2,5]]. Le modèle garde son défaut : plus le facteur de crête est grand, plus il favorise voix.

À quoi sert le biais si nous avons déjà des poids ? Il permet de décaler un score, même sans changer la contribution des mesures. Prenons un biais voix de 1 et gardons les poids identiques. Prévoyez d’abord le score voix de A.

Le nouveau score voix de A vaut 2,5. Le même décalage ajoute aussi 1 aux scores voix de B et C : ils deviennent 5,5 et 3,5. Le biais appartient à la règle commune, contrairement à notre retouche isolée de B.

Dans la suite, nous repartirons des biais nuls. Nous assemblerons notre propre nn.Module au moment de construire la boucle ; pour comprendre cette règle, appeler directement Linear suffit. Il manque surtout un moyen de quantifier son erreur sur B.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Linear possède deux entrées et deux sorties. X contient les trois extraits fictifs ; aucune classe n’est fournie dans ses colonnes.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
couche = nn.Linear(2, 2)


In [5]:
observer('X', X)
observer('couche', couche)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]


### Appliquer le contexte explicite

no_grad évite d’enregistrer cette affectation manuelle pour une dérivation. copy_ remplace les poids et zero_ fixe les deux biais à zéro.

In [6]:
with torch.no_grad():
    couche.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    couche.bias.zero_()


### Calculer l’état suivant

couche(X) réutilise les mêmes paramètres pour chaque ligne. Le résultat retrouve les scores déjà étudiés.

In [7]:
scores = couche(X)


In [8]:
observer('scores', scores)

scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [9]:
print("scores de depart :", scores)


scores de depart : tensor([[ 1.5000, -1.5000],
        [ 4.5000, -4.5000],
        [ 2.5000, -2.5000]], grad_fn=<AddmmBackward0>)


### Appliquer le contexte explicite

Le second contexte no_grad accompagne un autre réglage manuel. L’indice 0 sélectionne le biais voix ; les poids ne changent pas.

In [10]:
with torch.no_grad():
    couche.bias[0] = 1.


### Calculer l’état suivant

Nous recalculons les scores après l’affectation. Chaque score voix gagne 1, sans que les décisions sur nos trois extraits changent.

In [11]:
scores_avec_biais = couche(X)
decisions_avec_biais = scores_avec_biais.argmax(dim=1)


In [12]:
observer('scores_avec_biais', scores_avec_biais)
observer('decisions_avec_biais', decisions_avec_biais)

scores_avec_biais forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[2.5, -1.5], [5.5, -4.5], [3.5, -2.5]]
decisions_avec_biais forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 0, 0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("scores avec biais voix :", scores_avec_biais)
print("decisions :", decisions_avec_biais.tolist())

scores avec biais voix : tensor([[ 2.5000, -1.5000],
        [ 5.5000, -4.5000],
        [ 3.5000, -2.5000]], grad_fn=<AddmmBackward0>)
decisions : [0, 0, 0]


In [14]:
assert bool(torch.allclose(scores, torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]))), 'construire-un-module/main: torch.allclose(scores, torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]))'
assert bool(torch.allclose(scores_avec_biais[:, 0] - scores[:, 0], torch.ones(3))), 'construire-un-module/main: torch.allclose(scores_avec_biais[:, 0] - scores[:, 0], torch.ones(3))'
assert bool(torch.equal(scores_avec_biais[:, 1], scores[:, 1]) and decisions_avec_biais.tolist() == [0, 0, 0]), 'construire-un-module/main: torch.equal(scores_avec_biais[:, 1], scores[:, 1]) and decisions_avec_biais.tolist() == [0, 0, 0]'
assert bool(torch.equal(couche.weight, torch.tensor([[0., 1.], [0., -1.]])) and couche.bias.tolist() == [1., 0.]), 'construire-un-module/main: torch.equal(couche.weight, torch.tensor([[0., 1.], [0., -1.]])) and couche.bias.tolist() == [1., 0.]'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez le premier score d’une couche linéaire avec ces poids et ce biais.

In [15]:
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([0.,1.])
exo_b = torch.tensor(0.)
exo_score = ...  # Remplacer les points par votre réponse.
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score) - float((exo_x*exo_w).sum()+exo_b)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Faites les deux produits, puis leur somme et enfin le biais.

</details>

### Exercice 2

Gardez les mêmes poids [0,1], mais fixez le biais à 0,5. Quel est le nouveau score pour [.1,1.5] ?

In [16]:
exo_x=torch.tensor([.1,1.5])
exo_w=torch.tensor([0.,1.])
exo_b=torch.tensor(.5)
exo_score = ...  # Remplacer les points par votre réponse.
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score)-((exo_x*exo_w).sum()+exo_b).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le biais ajoute ici 0,5 au score déjà calculé. Il ne multiplie pas les mesures.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([0.,1.])
exo_b = torch.tensor(0.)
exo_score = 1.5
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score) - float((exo_x*exo_w).sum()+exo_b)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x=torch.tensor([.1,1.5])
exo_w=torch.tensor([0.,1.])
exo_b=torch.tensor(.5)
exo_score = 2.
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score)-((exo_x*exo_w).sum()+exo_b).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [19]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()